# Customer Churn Prediction with Tree-Based Models

This notebook compares Decision Tree, Random Forest, and Gradient Boosting models for telecom customer churn. It uses leakage-safe preprocessing, stratified validation, threshold analysis, and feature interpretation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV,
    cross_val_score, cross_val_predict
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier


## Helper functions


In [ ]:
def evaluate_model(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
    }

def print_metrics(title, metrics):
    print(f"\n{title}")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")

def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["No Churn", "Churn"]).plot()
    plt.title(title)
    plt.show()


## Data loading and cleaning

`TotalCharges` contains 11 blank strings. They correspond to customers with `tenure = 0`, so they are interpreted as customers who have not accumulated charges yet and are set to zero.


In [ ]:
df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(df["Churn"].value_counts(normalize=True))

blank_total_charges = df["TotalCharges"].str.strip().eq("")
print("Blank TotalCharges:", blank_total_charges.sum())
print(df.loc[blank_total_charges, ["tenure", "MonthlyCharges", "TotalCharges"]])

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.loc[df["tenure"] == 0, "TotalCharges"] = 0
assert df["TotalCharges"].isna().sum() == 0


## Exploratory analysis

The EDA focuses on churn rate, tenure, contract type, monthly charges, internet service, and tech support. These relationships are associative rather than causal.


In [ ]:
# Target distribution
churn_counts = df["Churn"].value_counts()
plt.figure(figsize=(6, 4))
plt.bar(churn_counts.index, churn_counts.values)
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Customers")
plt.show()

# Tenure by churn
plt.figure(figsize=(8, 5))
plt.hist(df.loc[df["Churn"] == "No", "tenure"], bins=20, alpha=0.6, label="No")
plt.hist(df.loc[df["Churn"] == "Yes", "tenure"], bins=20, alpha=0.6, label="Yes")
plt.title("Tenure Distribution by Churn")
plt.xlabel("Tenure (months)")
plt.ylabel("Customers")
plt.legend()
plt.show()

# Churn rate by contract
contract_churn = df.groupby("Contract")["Churn"].apply(lambda x: (x == "Yes").mean()).sort_values()
plt.figure(figsize=(7, 4))
plt.bar(contract_churn.index, contract_churn.values)
plt.title("Churn Rate by Contract Type")
plt.ylabel("Churn rate")
plt.show()

# Contract x InternetService interaction
interaction = (
    df.assign(ChurnFlag=(df["Churn"] == "Yes").astype(int))
      .groupby(["Contract", "InternetService"])["ChurnFlag"]
      .mean()
      .unstack()
)
interaction


## Train/test split and preprocessing

`customerID` is retained in the original dataframe but excluded from model features. Categorical variables are one-hot encoded inside a `ColumnTransformer` to avoid leakage.


In [ ]:
X = df.drop(columns=["Churn", "customerID"])
y = (df["Churn"] == "Yes").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

categorical_cols = X_train.select_dtypes(include="object").columns.tolist()
numeric_cols = X_train.select_dtypes(exclude="object").columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## Baseline and Decision Tree

The majority-class Dummy model establishes a baseline. An unrestricted Decision Tree is then used to illustrate overfitting, followed by the regularized configuration selected previously with cross-validation.


In [ ]:
# Dummy baseline
dummy_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", DummyClassifier(strategy="most_frequent")),
])
dummy_model.fit(X_train, y_train)
dummy_test_pred = dummy_model.predict(X_test)
dummy_test_prob = dummy_model.predict_proba(X_test)[:, 1]
dummy_metrics = evaluate_model(y_test, dummy_test_pred, dummy_test_prob)
print_metrics("DUMMY CLASSIFIER — TEST", dummy_metrics)

# Unrestricted tree
tree_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", DecisionTreeClassifier(random_state=42)),
])
tree_model.fit(X_train, y_train)
tree_train_pred = tree_model.predict(X_train)
tree_train_prob = tree_model.predict_proba(X_train)[:, 1]
tree_test_pred = tree_model.predict(X_test)
tree_test_prob = tree_model.predict_proba(X_test)[:, 1]
tree_train_metrics = evaluate_model(y_train, tree_train_pred, tree_train_prob)
tree_test_metrics = evaluate_model(y_test, tree_test_pred, tree_test_prob)
print_metrics("UNRESTRICTED TREE — TRAIN", tree_train_metrics)
print_metrics("UNRESTRICTED TREE — TEST", tree_test_metrics)
print("Depth:", tree_model.named_steps["model"].get_depth())
print("Leaves:", tree_model.named_steps["model"].get_n_leaves())

# Regularized tree — best configuration from 5-fold GridSearchCV
best_tree = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", DecisionTreeClassifier(
        random_state=42, class_weight="balanced", max_depth=5,
        min_samples_leaf=10, min_samples_split=2
    )),
])
best_tree.fit(X_train, y_train)
best_tree_test_pred = best_tree.predict(X_test)
best_tree_test_prob = best_tree.predict_proba(X_test)[:, 1]
best_tree_test_metrics = evaluate_model(y_test, best_tree_test_pred, best_tree_test_prob)
print_metrics("REGULARIZED TREE — TEST", best_tree_test_metrics)
plot_confusion(y_test, best_tree_test_pred, "Regularized Decision Tree — Confusion Matrix")


## Optional Decision Tree tuning

Set `RUN_TREE_TUNING = True` only when re-running the full hyperparameter search.


In [ ]:
RUN_TREE_TUNING = False

if RUN_TREE_TUNING:
    tree_tuning_pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", DecisionTreeClassifier(random_state=42)),
    ])
    tree_param_grid = {
        "model__max_depth": [3, 4, 5, 6, 8, 10],
        "model__min_samples_split": [2, 10, 20, 40],
        "model__min_samples_leaf": [1, 5, 10, 20, 40],
        "model__class_weight": [None, "balanced"],
    }
    tree_grid_search = GridSearchCV(
        tree_tuning_pipeline, tree_param_grid, scoring="roc_auc", cv=cv, n_jobs=-1, verbose=1
    )
    tree_grid_search.fit(X_train, y_train)
    print(tree_grid_search.best_params_)
    print("Best CV ROC-AUC:", tree_grid_search.best_score_)


## Random Forest

The final configuration below is the best setting found previously with 5-fold GridSearchCV. The optional tuning block can be enabled when a full re-search is needed.


In [ ]:
best_rf = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=10,
        max_features="sqrt", class_weight=None,
        random_state=42, n_jobs=-1
    )),
])
best_rf.fit(X_train, y_train)
best_rf_train_pred = best_rf.predict(X_train)
best_rf_train_prob = best_rf.predict_proba(X_train)[:, 1]
best_rf_test_pred = best_rf.predict(X_test)
best_rf_test_prob = best_rf.predict_proba(X_test)[:, 1]
best_rf_train_metrics = evaluate_model(y_train, best_rf_train_pred, best_rf_train_prob)
best_rf_test_metrics = evaluate_model(y_test, best_rf_test_pred, best_rf_test_prob)
print_metrics("TUNED RANDOM FOREST — TRAIN", best_rf_train_metrics)
print_metrics("TUNED RANDOM FOREST — TEST", best_rf_test_metrics)

rf_cv_scores = cross_val_score(best_rf, X_train, y_train, scoring="roc_auc", cv=cv, n_jobs=-1)
print("CV ROC-AUC:", rf_cv_scores)
print(f"Mean: {rf_cv_scores.mean():.4f} | Std: {rf_cv_scores.std():.4f}")

RUN_RF_TUNING = False
if RUN_RF_TUNING:
    rf_pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", RandomForestClassifier(random_state=42, n_jobs=-1)),
    ])
    rf_param_grid = {
        "model__n_estimators": [200, 500],
        "model__max_depth": [5, 8, 12, None],
        "model__min_samples_leaf": [1, 5, 10],
        "model__max_features": ["sqrt", 0.5],
        "model__class_weight": [None, "balanced"],
    }
    rf_grid_search = GridSearchCV(rf_pipeline, rf_param_grid, scoring="roc_auc", cv=cv, n_jobs=-1, verbose=1)
    rf_grid_search.fit(X_train, y_train)
    print(rf_grid_search.best_params_)
    print("Best CV ROC-AUC:", rf_grid_search.best_score_)


## Gradient Boosting

Gradient Boosting builds shallow trees sequentially. The selected model uses 200 depth-1 trees with a learning rate of 0.1.


In [ ]:
best_gb = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", GradientBoostingClassifier(
        learning_rate=0.1, max_depth=1, min_samples_leaf=10,
        n_estimators=200, random_state=42
    )),
])
best_gb.fit(X_train, y_train)
best_gb_train_pred = best_gb.predict(X_train)
best_gb_train_prob = best_gb.predict_proba(X_train)[:, 1]
best_gb_test_pred = best_gb.predict(X_test)
best_gb_test_prob = best_gb.predict_proba(X_test)[:, 1]
best_gb_train_metrics = evaluate_model(y_train, best_gb_train_pred, best_gb_train_prob)
best_gb_test_metrics = evaluate_model(y_test, best_gb_test_pred, best_gb_test_prob)
print_metrics("TUNED GRADIENT BOOSTING — TRAIN", best_gb_train_metrics)
print_metrics("TUNED GRADIENT BOOSTING — TEST", best_gb_test_metrics)
plot_confusion(y_test, best_gb_test_pred, "Tuned Gradient Boosting — Confusion Matrix")

RUN_GB_TUNING = False
if RUN_GB_TUNING:
    gb_pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", GradientBoostingClassifier(random_state=42)),
    ])
    gb_param_grid = {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [1, 2, 3],
        "model__min_samples_leaf": [5, 10],
    }
    gb_grid_search = GridSearchCV(gb_pipeline, gb_param_grid, scoring="roc_auc", cv=cv, n_jobs=-1, verbose=1)
    gb_grid_search.fit(X_train, y_train)
    print(gb_grid_search.best_params_)
    print("Best CV ROC-AUC:", gb_grid_search.best_score_)


## Model comparison


In [ ]:
model_results = pd.DataFrame({
    "Dummy": dummy_metrics,
    "Unrestricted Tree": tree_test_metrics,
    "Regularized Tree": best_tree_test_metrics,
    "Tuned Random Forest": best_rf_test_metrics,
    "Tuned Gradient Boosting": best_gb_test_metrics,
}).T
model_results


## Decision-threshold analysis

Thresholds are selected from out-of-fold probabilities on the training data rather than optimized directly on the test set.


In [ ]:
gb_oof_prob = cross_val_predict(
    best_gb, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, gb_oof_prob)
precision_t, recall_t = precision[:-1], recall[:-1]
f1_scores = 2 * precision_t * recall_t / (precision_t + recall_t + 1e-10)

max_f1_idx = np.argmax(f1_scores)
f1_threshold = thresholds[max_f1_idx]
print(f"Max-F1 threshold: {f1_threshold:.3f}")
print(f"OOF Precision: {precision_t[max_f1_idx]:.4f}")
print(f"OOF Recall: {recall_t[max_f1_idx]:.4f}")
print(f"OOF F1: {f1_scores[max_f1_idx]:.4f}")

target_recall = 0.75
valid = recall_t >= target_recall
best_recall_idx = np.argmax(precision_t[valid])
business_threshold = thresholds[valid][best_recall_idx]
print(f"Recall-oriented threshold: {business_threshold:.3f}")

gb_test_pred_f1 = (best_gb_test_prob >= f1_threshold).astype(int)
gb_test_pred_recall = (best_gb_test_prob >= business_threshold).astype(int)
gb_f1_threshold_metrics = evaluate_model(y_test, gb_test_pred_f1, best_gb_test_prob)
gb_recall_threshold_metrics = evaluate_model(y_test, gb_test_pred_recall, best_gb_test_prob)

threshold_results = pd.DataFrame({
    "GB — Threshold 0.50": best_gb_test_metrics,
    "GB — Max F1 Threshold": gb_f1_threshold_metrics,
    "GB — Recall-oriented Threshold": gb_recall_threshold_metrics,
}).T
threshold_results


## Feature interpretation

Tree-based feature importance is complemented with permutation importance on the original input columns. Importance is interpreted as predictive usefulness, not causality.


In [ ]:
fitted_preprocessor = best_gb.named_steps["preprocessor"]
gb_model = best_gb.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": gb_model.feature_importances_,
}).sort_values("Importance", ascending=False)
importance_df["Feature"] = (
    importance_df["Feature"].str.replace("cat__", "", regex=False).str.replace("num__", "", regex=False)
)

top_features = importance_df.head(15).sort_values("Importance")
plt.figure(figsize=(9, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("Gradient Boosting — Top Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

permutation_result = permutation_importance(
    best_gb, X_test, y_test, scoring="roc_auc", n_repeats=20, random_state=42, n_jobs=-1
)
permutation_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": permutation_result.importances_mean,
    "Std": permutation_result.importances_std,
}).sort_values("Importance", ascending=False)

top_permutation = permutation_df.head(15).sort_values("Importance")
plt.figure(figsize=(9, 6))
plt.barh(top_permutation["Feature"], top_permutation["Importance"])
plt.title("Gradient Boosting — Permutation Importance")
plt.xlabel("Decrease in ROC-AUC")
plt.tight_layout()
plt.show()

permutation_df
